# NB06 — Capa 3: Validación de Señales de Carga y Diseño de Corrección

## Dos objetivos metodológicamente separados

**Objetivo 3a — Validación empírica** *(componente académico)*  
¿Las métricas de carga semanal (ACWR, zonas de intensidad, exertion percibida) son
predictoras válidas de riesgo de lesión en corredores competitivos?

**Objetivo 3b — Diseño heurístico de Capa 3** *(componente producto)*  
¿Cómo comunicar el estado de carga del atleta en la predicción de rendimiento?
Esta sección es **explícitamente heurística** — no está calibrada sobre resultados de rendimiento.

---
> **⚠️ Advertencia metodológica central**: el dataset de Injury Prediction no contiene
> tiempos de carrera. Por lo tanto, **Capa 3 no puede ajustar numéricamente el estimado
> de ritmo**. Solo puede ajustar `confidence` y agregar un mensaje contextual (`load_note`).
> Cualquier afirmación más fuerte requeriría datos de rendimiento inexistentes en este dataset.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

INJURY_PATH = (PROJECT_ROOT / 'Datasets running'
               / 'Nuevo dataset  Injury Prediction for Competitive Runners'
               / 'week_approach_maskedID_timeseries.csv')
ANDRES_PATH = (PROJECT_ROOT / 'data' / 'athletes'
               / '1070982737' / 'features' / 'weekly_features.parquet')

print(f'Injury dataset exists : {INJURY_PATH.exists()}')
print(f'Andrés features exist : {ANDRES_PATH.exists()}')


---
## Sección 1: Descripción del dataset y EDA

**Fuente**: *Injury Prediction for Competitive Runners* (Rossi et al.) — 74 atletas competitivos.

**Estructura** (`week_approach`): cada fila = una semana para un atleta,
con ventana rodante de 3 semanas.
- Sin sufijo: semana actual (week 0)
- Sufijo `.1`: semana anterior (week 1)
- Sufijo `.2`: dos semanas atrás (week 2)
- Pre-calculados: `rel total kms week 0_1` = km₀ / km₁ (proxy ACWR ratio simple)
- Target: `injury` (0 = sin lesión, 1 = lesión reportada)

**Limitación estructural**: los ratios semanales simples (km₀/km₁) son una aproximación
del ACWR; no equivalen al ACWR EWMA de Hulin et al. (2016) que usa ventanas
de 7 días (ATL) y 28 días (CTL). La validación de `acwr_zone()` en esta sección
es aproximada por esta razón.


In [ ]:
ACWR_COL = 'rel total kms week 0_1'

df_raw = pd.read_csv(INJURY_PATH)
print(f'Forma original: {df_raw.shape}')

# Filtrar ACWR proxy extremo (semana previa ≈ 0 km → ratio infinito)
df = df_raw[df_raw[ACWR_COL] <= 5].copy()
print(f'Tras filtrar ACWR proxy > 5: {df.shape}')
print(f'Filas removidas (semana previa ≈ 0 km): {len(df_raw) - len(df)}')

df['intensity_ratio'] = df['total km Z5-T1-T2'] / (df['total kms'] + 0.001)
df['zone_34_ratio']   = df['total km Z3-4']      / (df['total kms'] + 0.001)

print(f'\nAtletas únicos     : {df["Athlete ID"].nunique()}')
print(f'Filas/atleta (media): {df.groupby("Athlete ID").size().mean():.1f}')


In [ ]:
n_total  = len(df)
n_injury = int(df['injury'].sum())
prev     = n_injury / n_total

print('=== Prevalencia de lesión ===')
print(f'Total observaciones : {n_total:,}')
print(f'Semanas con lesión  : {n_injury:,} ({prev:.1%})')
print(f'Semanas sin lesión  : {n_total - n_injury:,} ({1-prev:.1%})')
ath_inj = df.groupby('Athlete ID')['injury'].sum()
print(f'\nAtletas con ≥1 lesión: {(ath_inj > 0).sum()} / {len(ath_inj)}')
print(f'Lesiones/atleta lesionado (media): {ath_inj[ath_inj > 0].mean():.1f}')
print()
print('⚠️  Clase muy desbalanceada (1.3% positivos).')
print('    Accuracy es métrica trivial — evaluar con AUC-ROC.')


In [ ]:
bins   = [0, 0.8, 1.3, 1.5, 5.0]
labels = ['<0.8 BAJO', '0.8-1.3 OPTIMO', '1.3-1.5 PRECAUCION', '>1.5 ALTO']
df['acwr_zone_proxy'] = pd.cut(df[ACWR_COL], bins=bins, labels=labels)

zone_stats = (
    df.groupby('acwr_zone_proxy', observed=True)['injury']
      .agg(n='count', n_injury='sum', injury_rate='mean')
      .reset_index()
)
zone_stats['injury_rate_pct'] = zone_stats['injury_rate'] * 100

print('Injury rate por zona ACWR (umbrales de acwr_zone()):')
print(zone_stats[['acwr_zone_proxy', 'n', 'n_injury', 'injury_rate_pct']].to_string(index=False))
print()
print(f'Correlación Pearson ACWR proxy vs injury: '
      f'{df["injury"].corr(df[ACWR_COL]):+.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Boxplot ACWR por injury ---
ax = axes[0]
d0 = df.loc[df['injury'] == 0, ACWR_COL].clip(0, 3).values
d1 = df.loc[df['injury'] == 1, ACWR_COL].clip(0, 3).values
bp = ax.boxplot([d0, d1],
                labels=[f'Sin lesión\n(n={len(d0):,})', f'Lesión\n(n={len(d1):,})'],
                patch_artist=True, notch=False,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='black', linewidth=2))
for threshold, color, lbl in [(0.8, 'orange', '0.8'), (1.3, 'red', '1.3'),
                               (1.5, 'darkred', '1.5')]:
    ax.axhline(threshold, color=color, ls='--', alpha=0.7, label=f'Umbral {lbl}')
ax.set_ylabel('ACWR proxy (km₀ / km₁)')
ax.set_title('ACWR proxy por estado de lesión')
ax.legend(fontsize=7)

# --- Injury rate por zona ---
ax2 = axes[1]
zone_labels = zone_stats['acwr_zone_proxy'].astype(str).tolist()
rates       = zone_stats['injury_rate_pct'].tolist()
colors      = ['steelblue', 'green', 'orange', 'firebrick']
ax2.bar(zone_labels, rates, color=colors, alpha=0.75, edgecolor='black')
global_rate = df['injury'].mean() * 100
ax2.axhline(global_rate, color='gray', ls='--',
            label=f'Media global: {global_rate:.2f}%')
ax2.set_ylabel('Injury rate (%)')
ax2.set_title('Injury rate por zona ACWR (umbrales acwr_zone())')
ax2.tick_params(axis='x', rotation=10)
ax2.legend(fontsize=8)
ax2.set_ylim(0, max(rates) * 1.4)

plt.tight_layout()
plt.suptitle('Figura 1: ACWR proxy — distribución y tasa de lesión por zona',
             y=1.02, fontsize=10)
plt.savefig('fig_nb06_acwr_zones.png', dpi=120, bbox_inches='tight')
plt.show()
print('Observación clave: injury rate prácticamente PLANA (1.40% – 1.48%).')
print('La zona ACWR no discrimina riesgo de lesión en este dataset.')


### ⚠️ Limitación: ratio simple vs. ACWR EWMA

| Métrica | Definición | Ventana temporal |
|---|---|---|
| Ratio simple (dataset) | km₀ / km₁ | 2 semanas |
| ACWR Hulin EWMA | ATL₇d / CTL₂₈d | 7 días / 28 días continuos |
| ACWR en `load_metrics.py` | ATL(span=1) / CTL(span=6) semanal | EWMA discreto semanal |

El ratio simple amplifica variaciones semana a semana sin el efecto amortiguador del EWMA.
Si el ratio simple ya no muestra señal, no es razonable asumir que el ACWR EWMA
la mostraría sin historial diario completo.


---
## Sección 2: Señal de variables de carga sobre riesgo de lesión

Análisis bivariado: diferencia de medias y correlación de Pearson
(equivalente a correlación punto-biserial con target binario).

|r| < 0.10 = señal muy débil. |r| 0.10–0.30 = débil. |r| > 0.30 = moderada.


In [ ]:
features_map = {
    'acwr_proxy':        'rel total kms week 0_1',
    'total_kms':         'total kms',
    'km_Z5_T1_T2':       'total km Z5-T1-T2',
    'km_Z3_4':           'total km Z3-4',
    'intensity_ratio':   'intensity_ratio',
    'avg_exertion':      'avg exertion',
    'avg_recovery':      'avg recovery',
    'avg_train_success': 'avg training success',
    'nr_sessions':       'nr. sessions',
    'nr_tough_sessions': 'nr. tough sessions (effort in Z5, T1 or T2)',
    'nr_rest_days':      'nr. rest days',
}

rows = []
for name, col in features_map.items():
    s  = df[col].fillna(0)
    r  = float(df['injury'].corr(s))
    m0 = float(df.loc[df['injury'] == 0, col].mean())
    m1 = float(df.loc[df['injury'] == 1, col].mean())
    delta = (m1 - m0) / (abs(m0) + 1e-9) * 100
    rows.append({'variable': name, 'r': r, 'mean_no_inj': m0,
                 'mean_inj': m1, 'delta_pct': delta})

signal_df = pd.DataFrame(rows).sort_values('r', ascending=False)

print(f'{"variable":22s}  {"r":>8s}  {"mean_no_inj":>12s}  {"mean_inj":>10s}  {"delta%":>8s}')
print('-' * 70)
for _, row in signal_df.iterrows():
    print(f'{row.variable:22s}  {row.r:+8.4f}  {row.mean_no_inj:12.3f}  '
          f'{row.mean_inj:10.3f}  {row.delta_pct:+7.1f}%')
print()
top = signal_df.iloc[0]
print(f'Variable con mayor señal: {top.variable} (r={top.r:+.4f})')
print('Todas las correlaciones < 0.06 — señal MUY DÉBIL a nivel individual.')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = signal_df.sort_values('r')
bar_colors = ['firebrick' if r < 0 else 'steelblue' for r in plot_df['r']]
ax.barh(plot_df['variable'], plot_df['r'], color=bar_colors, alpha=0.75, edgecolor='black')
ax.axvline(0, color='black', lw=0.8)
for sign in [1, -1]:
    ax.axvline(sign * 0.10, color='gray', ls='--', alpha=0.5,
               label='|r|=0.10 (señal débil)' if sign == 1 else None)
ax.set_xlabel('Correlación Pearson (punto-biserial) con injury')
ax.set_title('Figura 2: Señal de variables de carga sobre riesgo de lesión')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('fig_nb06_signal.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Sección 3: Modelo de clasificación de riesgo de lesión

**Modelo**: Regresión logística con pesos de clase balanceados para compensar el desbalance 99:1.

**Validación**: Leave-One-Athlete-Out CV (LOAO-CV).  
Cada fold entrena con 73 atletas y evalúa en el restante.
Solo se incluyen folds donde el atleta de test tiene ≥ 1 lesión registrada.

**Métricas**: AUC-ROC (insensible al desbalance).  
AUC = 0.5 → azar. AUC = 1.0 → perfecto.

**Implementación**: numpy puro (descenso por gradiente, λ=0.01 L2, 300 iteraciones).


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20, 20)))

def logreg_fit(X_tr, y_tr, lr=0.05, lam=0.01, n_iter=300):
    """Regresión logística por gradiente con balanced class weights."""
    n_pos = max(y_tr.sum(), 1)
    n_neg = max(len(y_tr) - n_pos, 1)
    cw  = np.where(y_tr == 1, n_neg / n_pos, 1.0)
    Xb  = np.c_[np.ones(len(X_tr)), X_tr]
    w   = np.zeros(Xb.shape[1])
    for _ in range(n_iter):
        p    = sigmoid(Xb @ w)
        grad = Xb.T @ (cw * (p - y_tr)) / len(y_tr) + lam * w
        w   -= lr * grad
    return w

def roc_auc_numpy(y_true, y_score):
    """AUC-ROC via método Mann-Whitney (equivalente a trapezoide)."""
    order = np.argsort(y_score)[::-1]
    y_s   = y_true[order]
    n_pos = y_s.sum()
    n_neg = len(y_s) - n_pos
    if n_pos == 0 or n_neg == 0:
        return np.nan
    cum_tp = np.cumsum(y_s)
    return float((cum_tp[y_s == 0] - 0.5).sum() / (n_pos * n_neg))

FEAT_COLS = [
    'rel total kms week 0_1',
    'total km Z5-T1-T2',
    'avg exertion',
    'avg recovery',
    'intensity_ratio',
    'nr. sessions',
    'nr. tough sessions (effort in Z5, T1 or T2)',
]
FEAT_LABELS = ['ACWR proxy', 'km Z5-T1-T2', 'Avg exertion',
               'Avg recovery', 'Intensity ratio', 'Nr. sessions', 'Nr. tough sessions']

df_model = df.copy()
for col in FEAT_COLS:
    df_model[col] = df_model[col].fillna(0.0)

X_all = df_model[FEAT_COLS].values.astype(float)
y_all = df_model['injury'].values.astype(float)
g_all = df_model['Athlete ID'].values
athletes = np.unique(g_all)

results, all_coefs = [], []
for ath in athletes:
    mask_te = g_all == ath
    X_tr, X_te = X_all[~mask_te], X_all[mask_te]
    y_tr, y_te = y_all[~mask_te], y_all[mask_te]

    if y_te.sum() == 0:
        continue

    mu, sd  = X_tr.mean(0), X_tr.std(0) + 1e-9
    X_tr_s  = (X_tr - mu) / sd
    X_te_s  = (X_te - mu) / sd

    w    = logreg_fit(X_tr_s, y_tr)
    prob = sigmoid(np.c_[np.ones(len(X_te_s)), X_te_s] @ w)
    auc  = roc_auc_numpy(y_te, prob)

    results.append({'athlete': ath, 'n_test': len(y_te),
                    'n_injury': int(y_te.sum()), 'auc': auc})
    all_coefs.append(w[1:])

res_df = pd.DataFrame(results).dropna(subset=['auc'])
print(f'Folds evaluables (atleta con ≥1 lesión en test): {len(res_df)} / {len(athletes)}')
print()
print('=== Distribución AUC — LOAO-CV ===')
print(f'  Mediana AUC : {res_df.auc.median():.3f}')
print(f'  Media AUC   : {res_df.auc.mean():.3f}')
print(f'  Std AUC     : {res_df.auc.std():.3f}')
print(f'  P25 – P75   : {res_df.auc.quantile(0.25):.3f} – {res_df.auc.quantile(0.75):.3f}')
print(f'  Min / Max   : {res_df.auc.min():.3f} / {res_df.auc.max():.3f}')
print()
print('Interpretación:')
print('  AUC < 0.55         → prácticamente azar')
print('  AUC 0.55 – 0.65    → señal débil')
print('  AUC 0.65 – 0.75    → señal moderada')
print('  AUC > 0.75         → señal sólida')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución AUC por fold
ax = axes[0]
ax.hist(res_df['auc'], bins=15, color='steelblue', edgecolor='black', alpha=0.75)
ax.axvline(res_df['auc'].median(), color='red', ls='--',
           label=f'Mediana: {res_df["auc"].median():.3f}')
ax.axvline(0.5, color='gray', ls=':', label='Azar (0.5)')
ax.set_xlabel('AUC-ROC por fold')
ax.set_ylabel('Número de atletas')
ax.set_title('Distribución AUC — LOAO-CV')
ax.legend(fontsize=8)

# Coeficientes medios
ax2 = axes[1]
coef_matrix = np.array(all_coefs)
coef_mean   = coef_matrix.mean(0)
coef_std    = coef_matrix.std(0)
order       = np.argsort(coef_mean)
bar_c       = ['firebrick' if c < 0 else 'steelblue' for c in coef_mean[order]]
ax2.barh([FEAT_LABELS[i] for i in order], coef_mean[order],
         xerr=coef_std[order], color=bar_c, alpha=0.75,
         edgecolor='black', ecolor='gray', capsize=3)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_xlabel('Coeficiente medio ± SD entre folds (escala estandarizada)')
ax2.set_title('Importancia de features — Regresión Logística LOAO-CV')

plt.tight_layout()
plt.suptitle('Figura 3: Modelo de clasificación LOAO-CV', y=1.02, fontsize=10)
plt.savefig('fig_nb06_model.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Sección 4: Validación de umbrales de `acwr_zone()`

`acwr_zone()` en `src/ml/load_metrics.py` usa umbrales de Hulin et al. (2016):

| Zona | Rango ACWR | Expectativa |
|---|---|---|
| BAJO | < 0.8 | subentrenamiento — riesgo bajo |
| OPTIMO | 0.8 – 1.3 | zona segura |
| PRECAUCION | 1.3 – 1.5 | riesgo creciente |
| ALTO | > 1.5 | riesgo elevado |

**Pregunta**: ¿los datos reproducen el patrón monotónico creciente de injury rate?


In [ ]:
print('=== Validación de umbrales acwr_zone() ===')
print()
print(f'{"Zona":22s} {"N obs":>7s} {"N lesión":>9s} {"Injury rate":>12s} {"Patrón esperado":>20s}')
print('-' * 75)
zone_expected = {
    '<0.8 BAJO':          'creciente → tope',
    '0.8-1.3 OPTIMO':     'mínimo',
    '1.3-1.5 PRECAUCION': 'subida',
    '>1.5 ALTO':          'máximo',
}
for _, row in zone_stats.iterrows():
    zone = str(row['acwr_zone_proxy'])
    exp  = zone_expected.get(zone, '')
    print(f'{zone:22s} {int(row.n):7d} {int(row.n_injury):9d}'
          f' {row.injury_rate_pct:11.2f}%  {exp:>20s}')

print()
max_rate = zone_stats['injury_rate_pct'].max()
min_rate = zone_stats['injury_rate_pct'].min()
print(f'Rango injury rate entre zonas: {min_rate:.2f}% – {max_rate:.2f}%')
print(f'Diferencia máxima: {max_rate - min_rate:.2f}pp')
print()
print('HALLAZGO: La injury rate es prácticamente PLANA.')
print('El patrón monotónico BAJO→ALTO NO se reproduce en este dataset.')
print('Los umbrales de acwr_zone() son heurísticos — no validados aquí.')
print()
print('Posibles explicaciones:')
print('  1. Ratio simple km₀/km₁ ≠ ACWR EWMA (la métrica que Hulin usó)')
print('  2. Desfase temporal: lesión ocurre semanas después del pico de carga')
print('  3. Atletas competitivos tienen mayor tolerancia a cargas elevadas')


---
## Sección 5: Diseño de Capa 3 — Heurística de corrección de confianza

> **⚠️ Esta sección es explícitamente heurística.**  
> El análisis de las secciones anteriores muestra que la señal empírica de ACWR
> sobre lesión es débil e inestable. El diseño de Capa 3 **no puede presentarse**
> como "empíricamente validado" desde este dataset. Se justifica como heurística
> razonada para un producto de coaching, con disclaimer explícito al usuario.

**Principio de diseño**: un atleta con carga elevada tiene mayor incertidumbre
sobre su rendimiento real (puede estar más fatigado de lo que sus PRs sugieren).
Este principio es razonable conceptualmente, aunque no esté calibrado en datos propios.

**Restricción hard**: el rango numérico `pace_range_fmt` **no se modifica**.
Solo se ajustan `confidence` y `load_note`.


In [ ]:
CAPA3_DESIGN = [
    dict(zona='BAJO',         acwr='< 0.8',      ajuste='ninguno',
         load_note='Volumen bajo. Predicción válida; rendimiento puede estar '
                   'por encima si hubo buen descanso.'),
    dict(zona='OPTIMO',       acwr='0.8 – 1.3',  ajuste='ninguno',
         load_note=None),
    dict(zona='PRECAUCION',   acwr='1.3 – 1.5',  ajuste='baja 1 nivel',
         load_note='Carga aguda elevada. Rendimiento actual puede estar '
                   'afectado por fatiga acumulada.'),
    dict(zona='ALTO',         acwr='> 1.5',       ajuste='fuerza a BAJA',
         load_note='Carga de alto riesgo. Priorizar recuperación. '
                   'Predicción de rendimiento con baja fiabilidad.'),
]

print('=== Capa 3 — Mapeo zona → ajuste de confidence ===')
print(f'  {"Zona":12s} {"ACWR":10s} {"Ajuste confidence":20s} load_note')
print('-' * 90)
for r in CAPA3_DESIGN:
    note = f'"{r["load_note"][:50]}..."' if r['load_note'] else 'None'
    print(f'  {r["zona"]:12s} {r["acwr"]:10s} {r["ajuste"]:20s} {note}')

print()
print('Ejemplo JSON con Capa 3 activa (zona PRECAUCION):')
print('  {')
print('    "pace_range_fmt":             "5:10 – 5:30 min/km",  // NO cambia')
print('    "confidence":                 "MEDIA-BAJA",           // degradado de MEDIA')
print('    "load_zone":                  "PRECAUCION",')
print('    "load_acwr":                  1.42,')
print('    "load_confidence_adjustment": "degraded",')
print('    "load_note":                  "Carga aguda elevada..."')
print('  }')


### Tabla de separación: empírico vs. heurístico

| Componente | Estado | Fuente |
|---|---|---|
| Señal avg_exertion → lesión | Débil (r=+0.048) | Este dataset (NB06) |
| Señal km Z5-T1-T2 → lesión | Débil (r=+0.022) | Este dataset (NB06) |
| ACWR ratio → lesión | Sin evidencia (r=+0.011, rate plana) | Este dataset (NB06) |
| Umbrales 0.8 / 1.3 / 1.5 | Heurístico | Literatura (Hulin 2016); no replicado aquí |
| Magnitud del ajuste de confidence | Heurístico | Diseño razonado — sin calibración |
| Corrección numérica del ritmo | No implementada / No justificable | Sin datos de rendimiento |

**Capa 3, si se activa, debe presentarse al usuario como señal de contexto,
no como corrección del modelo predictivo.**


---
## Sección 6: Aplicación ilustrativa — Andrés Restrepo

Carga real desde Strava (últimas semanas disponibles).
Calcula ACWR EWMA con `load_metrics.py` y clasifica la zona actual.

*Este bloque requiere el pipeline ejecutado localmente.*


In [ ]:
from src.ml.load_metrics import add_load_metrics, acwr_zone

if not ANDRES_PATH.exists():
    print('⚠️  weekly_features.parquet no disponible en este entorno.')
    print('    Ejecuta primero: python -m pipeline.run --cedula 1070982737')
else:
    import duckdb
    wf = duckdb.query(f"SELECT * FROM '{ANDRES_PATH.as_posix()}'").df()
    wf = wf.sort_values('week_start').reset_index(drop=True)

    load_col = 'km_total' if 'km_total' in wf.columns else 'distance'

    if load_col not in wf.columns:
        print(f'⚠️  Columna de carga no encontrada. Columnas: {list(wf.columns)}')
    else:
        wf = add_load_metrics(wf, load_col=load_col, granularity='weekly')
        last8 = wf.tail(8)[['week_start', load_col, 'ctl', 'atl', 'tsb', 'acwr']].copy()
        last8['zona'] = last8['acwr'].apply(acwr_zone)

        print('=== ACWR semanal — Andrés Restrepo (últimas 8 semanas) ===')
        print(last8.to_string(index=False))

        latest  = last8.iloc[-1]
        zona    = latest['zona']
        print(f'\nEstado actual: zona {zona}  |  ACWR = {latest["acwr"]:.3f}')

        ajuste = {
            'BAJO': 'ninguno', 'OPTIMO': 'ninguno',
            'PRECAUCION': 'baja 1 nivel', 'ALTO': 'fuerza a BAJA',
            'SIN_DATOS': 'sin ajuste (sin datos)'
        }.get(zona, 'sin ajuste')
        print(f'Capa 3 (si activa): confidence → {ajuste}')


In [ ]:
# ─── Conclusiones NB06 — 6 preguntas ────────────────────────────────────────
print('=' * 72)
print('CONCLUSIONES NB06')
print('=' * 72)

median_auc = res_df['auc'].median()
std_auc    = res_df['auc'].std()
p25, p75   = res_df['auc'].quantile([0.25, 0.75]).values

top_row    = signal_df.iloc[0]
top_var    = top_row['variable']
top_r      = top_row['r']
acwr_r     = float(signal_df.loc[signal_df.variable == 'acwr_proxy', 'r'].values[0])

print()
print('1. ¿Qué variables mostraron señal real y con qué magnitud?')
print(f'   Mayor señal: {top_var} (r={top_r:+.4f}, delta +{top_row["delta_pct"]:.0f}% injury vs no-injury).')
print(f'   km Z5-T1-T2 e intensity_ratio también muestran señal positiva (r≈+0.02).')
print(f'   ACWR ratio simple: prácticamente nulo (r={acwr_r:+.4f}, injury rate plana por zona).')
print(f'   TODAS las |r| < 0.06 — señal muy débil a nivel individual.')

print()
print('2. ¿Qué tan estable o inestable fue el LOAO-CV?')
print(f'   Mediana AUC = {median_auc:.3f} | Std = {std_auc:.3f} | P25-P75 = [{p25:.3f} – {p75:.3f}]')
print(f'   Rango: {res_df.auc.min():.3f} – {res_df.auc.max():.3f}.')
print(f'   ALTAMENTE INESTABLE. Algunos atletas AUC > 0.80; otros AUC < 0.50.')
print(f'   El modelo no generaliza de forma confiable entre atletas.')

print()
print('3. ¿Los umbrales de acwr_zone() son compatibles con la evidencia?')
max_r2 = zone_stats['injury_rate_pct'].max()
min_r2 = zone_stats['injury_rate_pct'].min()
print(f'   NO. Injury rate plana entre zonas ({min_r2:.2f}% – {max_r2:.2f}%).')
print(f'   Patrón monotónico creciente (BAJO→ALTO) no reproducido.')
print(f'   Umbrales de acwr_zone() son heurísticos (Hulin 2016), no validados aquí.')

print()
print('4. ¿La evidencia justifica una Capa 3 que ajuste confidence?')
print('   PARCIALMENTE. La evidencia empírica no permite calibrar la magnitud del ajuste.')
print('   El principio conceptual (mayor carga = mayor incertidumbre) es razonable.')
print('   Capa 3 puede existir como señal de contexto con disclaimer explícito.')

print()
print('5. ¿Qué queda empírico y qué queda heurístico?')
print('   Empírico (este dataset): señal débil de avg_exertion e intensidad.')
print('   Empírico (NB03/NB05b): calibración Riegel — Capas 0-2, sólido.')
print('   Heurístico: umbrales ACWR, magnitud del ajuste de confidence.')
print('   No justificable: corrección numérica del ritmo estimado.')

print()
print('6. ¿Recomendación: activar Capa 3 o mantenerla apagada?')
print('   MANTENER APAGADA en producción.')
print('   Razones:')
print('     - Evidencia insuficiente para justificar ajuste de confidence calibrado.')
print('     - ACWR ratio no discrimina lesión en este dataset.')
print('     - Modelo LOAO-CV inestable (Std AUC = 0.16).')
print('   Para la tesis: documentar como "Capa 3: diseño propuesto" con fundamento')
print('   teórico y limitaciones empíricas explícitas. Activar solo con datos de')
print('   rendimiento post-carga del atleta real.')
print()
print('=' * 72)


---
## Declaración metodológica final

### ¿Qué aporta NB06 a la tesis?

1. **Aplica correctamente** Leave-One-Athlete-Out CV — única estrategia válida
   para series temporales por atleta sin data leakage.

2. **Reporta honestamente** que el ACWR volumétrico no discrimina lesión en este
   dataset y que el modelo de clasificación es altamente inestable (Std AUC = 0.16).

3. **Distingue con precisión** entre lo que la literatura establece (Hulin et al.),
   lo que los datos confirman (señal débil de exertion e intensidad) y
   lo que sigue siendo heurístico (umbrales, magnitud del ajuste).

4. **Diseña Capa 3** como señal de contexto (no corrección calibrada), preservando
   la integridad metodológica de la tesis.

### Estado del sistema de predicción

| Capa | Estado | Evidencia |
|---|---|---|
| Capa 0: selección PR | ✅ operativa | — |
| Capa 1: Riegel calibrado | ✅ operativa | Boston 2015-2018, n=102K (NB03) |
| Capa 2: corrección demográfica | ✅ operativa | Results.csv, n=429K (NB04) |
| Capa 2b: step multipliers validados | ✅ validados | Boston splits, n=99K (NB05b) |
| Capa 3: corrección por carga | 📄 diseñada, **apagada** | evidencia insuficiente (NB06) |
| Capa 4: wellness / check-in | 🔲 no implementada | pendiente |
